# 02 — Model inspection and random predictions
Builds the lightweight model for the shared selected dataset and performs labelled random-weight checks on training clips only.

In [1]:
from pathlib import Path
import subprocess
import sys
REPO_ROOT = Path('/content/phd_research')
if not REPO_ROOT.is_dir():
    subprocess.run(['git', 'clone', 'https://github.com/sanelehlabisa/phd_research.git', str(REPO_ROOT)], check=True)
else:
    checkout_dirty = subprocess.run(['git', 'status', '--porcelain'], cwd=REPO_ROOT, capture_output=True, text=True, check=True).stdout.strip()
    if checkout_dirty:
        print('Existing Colab checkout has local changes; preserving them and skipping update.')
    else:
        subprocess.run(['git', 'pull', '--ff-only', 'origin', 'master'], cwd=REPO_ROOT, check=True)
IMPLEMENTATION_ROOT = REPO_ROOT / 'papers/001-journal-abnormal-activity-recognition/implementation'
bootstrap = subprocess.run([sys.executable, str(IMPLEMENTATION_ROOT / 'src/colab_bootstrap.py'), str(IMPLEMENTATION_ROOT / 'requirements.txt')])
if bootstrap.returncode == 75:
    raise SystemExit('Restart the Colab runtime, reconnect to the A100, and rerun from the top.')
bootstrap.check_returncode()
if str(IMPLEMENTATION_ROOT) not in sys.path:
    sys.path.insert(0, str(IMPLEMENTATION_ROOT))
import importlib
for module_name in [name for name in sys.modules if name == 'src' or name.startswith('src.')]:
    del sys.modules[module_name]
importlib.invalidate_caches()


In [2]:
from src.notebook_utils import inspect_random_model, printable, validate_runtime
print(printable(validate_runtime(require_cuda=True)))


{
  "cuda_available": true,
  "cuda_device": "NVIDIA A100-SXM4-80GB",
  "cuda_version": "12.8",
  "cudnn_version": 91002,
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.39",
  "python": "3.13.15",
  "pytorch": "2.9.1+cu128",
  "torchvision": "0.24.1+cu128"
}


In [3]:
inspection = inspect_random_model(IMPLEMENTATION_ROOT)
print(printable(inspection))


Using Colab cache for faster access to the 'violence-detection-dataset' dataset.
✅ 350 videos | mode=video | 2 classes: ['non-violent', 'violent']
{
  "classes": [
    "non-violent",
    "violent"
  ],
  "dataset": "vdd",
  "evidence_role": "random_weight_pipeline_check_only",
  "input_shape": [
    3,
    16,
    3,
    32,
    32
  ],
  "labels": [
    0,
    0,
    0
  ],
  "logit_shape": [
    3,
    2
  ],
  "model": {
    "dropout": 0.5,
    "hidden_classifier_width": null,
    "input_channels": 3,
    "layers": [
      {
        "filters": 8,
        "kernel_size": [
          3,
          3
        ]
      },
      {
        "filters": 16,
        "kernel_size": [
          3,
          3
        ]
      }
    ],
    "model_name": "CustomConvLSTM",
    "num_classes": 2
  },
  "predictions": [
    0,
    0,
    0
  ],
  "test_access": "locked",
  "trainable_parameters": 17122
}


## Five labelled random-weight predictions
Correctness here checks wiring only. Random weights are not trained performance or research evidence.

In [ ]:
inspection = inspect_random_model(IMPLEMENTATION_ROOT)
assert len(inspection['examples']) == 5
for number, example in enumerate(inspection['examples'], start=1):
    status = 'CORRECT' if example['correct'] else 'WRONG'
    print(f"{number}. {status} | expected={example['expected']} | predicted={example['predicted']} | confidence={example['confidence']:.3f}")
